# 第5回: Memory Management

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session05/session05_memory.ipynb)

エージェントは複雑なタスクや一貫した体験のために過去のやり取りを覚える必要がある。メモリが無いとステートレスになり、文脈維持・経験からの学習・パーソナライズができず、単純な一問一答に留まる。核心は、単一会話の即時・一時情報と、時間をかけて蓄積する膨大で永続的な知識の両方を、どう効果的に管理するか。


---
## 0. 環境準備

In [ ]:
%pip install -q langchain langchain-core langchain-openai langgraph

In [ ]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY を入力する: ")

print("APIキー設定完了" if os.environ.get("OPENAI_API_KEY") else "未設定")

---
# 1. 前回の復習: ReAct

- 推論と行動を繰り返すだけの単純なアルゴリズム
- 行動の必要がなくなるまで繰り返す
- 最後の推論結果が最終レスポンス
- 1ターンのみの応答

[![](https://mermaid.ink/img/pako:eNp1U8GOmzAQ_RVrriVJk0AJPmyrtteoh24vDauVY08CKtjINk1Tln-v7QSWNFpf8Lz37Jl5gzvgSiBQOFTqxAumba5zSdxqDernUjat3S1-uD3ZojHsiIunQWHOxmL9XF_w3eJ7iO91tctQ7Q6MHthMq72yZOuRkWdCfOqIKViDlPBS84hUbI8VJTm8y4H0g5ArKUpbKtlZpSrCWVWZjyPrsWsWwzWehC5_o56dNEpekEfHjhlVa0Nf38J30pFVzWspPupfKefNlHNhIG8EZDZ7mDh372UQuIbHU-3-qFlTEOeYHPWi1Mh9o-Tx84D5Nd3fuv__vVdfAxz8nxIBCNTo6JQeQSc5o_E67-1bhYRZ3OVBKe7G5lUvUr1c_b-dRrjCWz6YeqEhgqMuBVCrW4ygRl0zH0Ln2RxsgTXm4P8VwfSvHHLZuzMNkz-VqodjWrXHYgjaRjCLX0vmnHeKA6uMl7iKUX9RrbRA18tluANoB3-Axkk2T9M4STabOM1W6ySCM9BkNY-zLEtXWRKvs_XyQx_B35D0_XyTOg26tpXeXh5YeGf9P_iCHHE?type=png)](https://mermaid.live/edit#pako:eNp1k0FzmzAQhf-KZq_FdmqgGB3aTpKrp4eml5pORpbWhglIjCTquoT_XkkOBI-nXNC-96SVPlAPXAkECodanXjJtC10IYl7OoP6uZJtZ3erH25MtmgMO-Lq15gwZ2OxeW4u-m71PdS3ucZ1qHcHRg9sodVeWbL1yuQzIb72xJSsRUp4pXlEarbHmpICPhRAhjHIlRSVrZTsrVI14ayuzZfJ9dpbF8M1noSufqNenDRKXpIn504dVWfDub6F9-xEVrXvW_HV8G45NnPPlcG8CpDF4vOM3OR1-6NmbUkcFzmpotLI_XHI0_2o-Wc-vmYclne45glXBjlQnhtBCNbEbW5Poouc0ficJ_i_jQTiN31QipuP41OvUr2-Ub5mHpbwYEd0FxsiOOpKALW6wwga1A3zJfTeLcCW2GAB_o8QTL8UUMjBzWmZ_KlUM07TqjuWY9G1gll8rJgj7xIHVhsfcTtG_aA6aYHGd3FYA2gPf4Amab7MsiRNN5sky9dxGsEZaLpeJnmeZ-s8TeI8_vhpiOBvaHq33GQug-7YSm8v1yjcpuEfjJUUCw)

---
# 2. 短期記憶と長期記憶

知的なエージェントが情報を保持するには、効果的なメモリ管理が欠かせない。人間と同じく、エージェントも目的に応じて異なる種類の記憶を必要とする。エージェントにおける「メモリ」とは、過去のやり取り・観測・学習から得た情報を保持して活用する能力を指し、これにより一貫した判断・文脈維持・継続的な改善が可能になる。記憶は大きく2種類に分けられる。

- **短期記憶（コンテキストメモリ）**: ワーキングメモリに相当し、いま処理中／直近の情報を保持する。LLM ベースのエージェントでは主に **コンテキストウィンドウ** の中に存在する。直近のメッセージ・エージェントの応答・ツールの実行結果・内省などが含まれ、次の応答や行動を方向づける。容量に上限があり、セッションが終わると失われる。古い会話を要約するなどして限られた枠を有効に使う。
- **長期記憶（永続メモリ）**: やり取り・タスク・期間をまたいで保持したい情報の置き場所。データはエージェントの外（DB・ナレッジグラフ・ベクトルDBなど）に保存する。ベクトルDBでは情報を数値ベクトルに変換して保存し、キーワード一致ではなく**意味的類似度（semantic search）**で取り出せる。必要なときに外部ストレージへ問い合わせ、取得した情報を短期コンテキストへ統合して使う。

---
# 3. 短期記憶の実装

## 3.1 ReActをLangGraphで再実装

前回は ReAct（推論と行動のループ）を手書きの `for` ループで実装した。ここではプログラムの見通しをよくするために、**LangGraph** を使って同じ振る舞いを再実装する。

- **LangGraph は LangChain 上に構築された、ステートフルなエージェントを作るためのフレームワーク。** LangChain がモデル・ツール・プロンプトといった「部品」を提供するのに対し、LangGraph はそれらを組み合わせた「制御フロー（どの順序で・どの条件で何を実行するか）」を宣言的に記述することに特化している。
- **LangGraph ではエージェントの振る舞いをグラフ（ノードとエッジから構成される構造）で表現する。** 主な構成要素は次の3つ。
  - **State（状態）**: グラフ全体で共有される作業データ。ここでは会話履歴 `messages` を持たせ、各ノードが返したメッセージが自動で積み上がっていく。これが ReAct における会話履歴そのものになる。
  - **Node（ノード）**: 実際の処理を行う関数。「LLM に推論させるノード（`llm`）」と「ツールを実行するノード（`tools`）」を用意する。各ノードは State を受け取り、更新分を返す。
  - **Edge（エッジ）**: ノード間の遷移。通常のエッジに加え、**条件分岐エッジ**を使うと「LLM がツール呼び出しを要求したか」を見て次の遷移先を動的に切り替えられる。
- 短期記憶は **checkpointer**（状態の保存機構）を使用して実装する。これを差し込むだけで State をスレッド単位で永続化し、ターンをまたいで記憶を引き継げる。

[![](https://mermaid.ink/img/pako:eNqNkM1ugzAMx18F-TgB2iBZSA67tNe9wJppipqMouajMmFai3j3BiaEttN88cf_Z1v2CMegDQjoo4pm36kWlSu-KonSZ8kOD-9ZUbxkR2Xth0uoXZWtsgExBNv_ApbK_yakVavy10MOLXYaRMTB5OAMOjWnMM66hHgyzkgQKdQKzxKkn1LPRfm3ENzahmFoT2syXPR2MIhPZfsZMV4b3IXBRxA1pcsMECN8gyCUl4wRSpuGMF7VSbyCoFVJOOes4pTUvH56nnK4LUsfy4YlxuguBnz9-fLy7OkO74Zzrg?type=png)](https://mermaid.live/edit#pako:eNqNUMtOxDAM_JVqjqit2LYhTQ5c4MoPQBCKNqFbkccqTRFQ9d9Jg6oVnPDFHs-MLXvB0SsNjinKqO9HOQRpq_dGBOGKFE9Xz0VV3RZHacyLTVKzM5dOFkTvzfSLy53_mdOWnfmbUWIIowKPYdYlrA5WbhDLxgvEk7ZagKdSyfAmINyaPGfpHr23uy34eTjtYD6ry63gr9JMm0Q7pcOdn10Eb8khzwBf8AHeEVZT2hHS9x1lTUtKfIKTpu4YY7RhpGtZe7hZS3zlpdd1T5NGqzH68PDz4Pzn9RsrtXGv)

In [ ]:
# Tool定義
import ast, operator, datetime as _dt
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

# --- 複数のツールを用意する ---
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}

def _safe_eval(node):
    if isinstance(node, ast.Expression): return _safe_eval(node.body)
    if isinstance(node, ast.Constant): return node.value
    if isinstance(node, ast.BinOp): return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp): return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("数式として解釈できない")

@tool(parse_docstring=True)
def calculator(expression: str) -> str:
    """数式を計算して結果を返す。四則演算・べき乗(**)・括弧に対応。

    Args:
        expression: 計算したい数式の文字列。例 '2 * (3 + 4)'。
    """
    try:
        return str(_safe_eval(ast.parse(expression, mode="eval")))
    except Exception as e:
        return f"計算エラー: {e}"

@tool
def current_datetime() -> str:
    """現在のローカル日時をISO形式で返す。"""
    return _dt.datetime.now().isoformat(timespec="seconds")

tools = [calculator, current_datetime]
print("用意したツール:", [t.name for t in tools])

In [ ]:
# State（状態）
from langgraph.graph.message import add_messages
from typing import Annotated, TypedDict

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
# Node
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import ToolNode

model = ChatOpenAI(
    model="gpt-5.4-mini",
    temperature=0,
)

model_with_tools = model.bind_tools(tools)

system_message = SystemMessage(
    content=(
    "あなたは有能なアシスタントです。"
    "計算は推測せず必ずツールを使い、最後は日本語で簡潔に答えてください。"
    )
)

# call_model
def call_model(state: AgentState):
    response = model_with_tools.invoke(
        [system_message] + state["messages"]
    )
    return {"messages": [response]}

# call_tools
call_tools = ToolNode(tools)

In [ ]:
# Edges & Graph
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import tools_condition

builder = StateGraph(AgentState)

# nods
builder.add_node(call_model)
builder.add_node("call_tools", call_tools)

# edges
builder.add_edge(START, "call_model")
builder.add_conditional_edges(
    "call_model",
    tools_condition,
    {
        "tools": "call_tools",
        END: END,
    },
)
builder.add_edge("call_tools", "call_model")

graph_without_memory = builder.compile()

In [ ]:
for chunk in graph_without_memory.stream(
    {
        "messages": [
            HumanMessage(
                content="2010年に建てられたマンションは、今年で築何年になりますか？"
            )
        ]
    },
    stream_mode="updates",
):
    print(chunk)
    print("-" * 80)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
graph_with_short_term_memory = builder.compile(
    checkpointer=checkpointer
)

In [ ]:
config = {
        "configurable": {
            "thread_id": "short-term-memory-demo-001", # checkpointerによって参照されるStateの保存名前空間の最上位
            # "checkpoint_ns": "", # スレッド内のサブ名前空間（サブグラフで使用）
            # "checkpoint_id": "<過去のID>" # チェックポイントごとに一意に割り振られるIDを直接指定（time travelやbranchingで使用）
        }
    }

for chunk in graph_with_short_term_memory.stream(
    {
        "messages": [
            HumanMessage(
                content="2010年に建てられたマンションは、今年で築何年になりますか？"
            )
        ]
    },
    # 同じグラフ定義を使いまわし、異なる実行設定(RunnableConfig)を差し替えることで会話の宛先（スレッド）等の実行コンテキストを変更する
    config = config
):
    print(chunk)
    print("-" * 80)

In [ ]:
for chunk in graph_with_short_term_memory.stream(
    {
        "messages": [
            HumanMessage(
                content="築40年になるのは西暦何年ですか？"
            )
        ]
    },
    config=config
):
    print(chunk)
    print("-" * 80)

In [ ]:
# 状態の確認
state = graph_with_short_term_memory.get_state(
    config=config
)
for m in state.values["messages"]:
    m.pretty_print()

---
# 4. 長期記憶の実装

長期記憶は会話をまたいで情報を保持し、より深い文脈とパーソナライズを与える。人間の記憶になぞらえて3種類に分けられる。

- **意味記憶（Semantic Memory）: 事実を覚える**: ユーザー設定やドメイン知識など具体的な事実・概念を保持する。応答の根拠付けに使い、パーソナルで関連性の高い対話を実現する。継続的に更新するユーザー「プロファイル」（JSON）として、または個々の事実文書の「コレクション」として管理する。
- **エピソード記憶（Episodic Memory）: 経験を覚える**: 過去の出来事や行動を思い出す。AIエージェントではタスクの達成方法を覚えるのに使われ、few-shot 例プロンプトとして実装されることが多い。
- **手続き記憶（Procedural Memory）: ルールを覚える**: タスクの遂行方法＝エージェントの中核的な指示や振る舞いで、システムプロンプトに含まれることが多い。エージェントが自分のプロンプトを修正して適応・改善することもある。有効な手法が **Reflection（内省）** で、現在の指示と直近のやり取りを与えて、自分の指示を洗練させる。

- 長期記憶は記憶すべき内容の抽出やバリデーションをしたうえで明示的な保存処理が必要
- 毎ターンの最初にロードし、最後に保存する

[![](https://mermaid.ink/img/pako:eNqNks9uhCAQxl9lM8dGjawKyKGX9toXaG02RKhrCrJB3O7W-O5VGuKuvZQLzDe_-fg3I9RGSGDQO-7kc8sby3V83le26nbzeHt438Xx404ZLg5aamOvIXUjeaTmSh307KYCsSor4IxR_R3glf85yIuzvHabg9yrHjxz1Yr5QhtyI3v0y7Z_uFvNQ_MzhFyYIYLGtgKYs4OMQEur-RLCuOQrcEepZQVsXgpuPyuoummuOfHu1RgdyqwZmmMIhpNYPwHYB1f9gshOSPtkhs4BK5G3ADbCBRjCJKFFnqMCI4rTAmcRXIFlKCGU4CxNU4JRhskUwbffNE1oWWQUlZSkZU5JuY9AitYZ-_LbBr4bph9o96vK?type=png)](https://mermaid.live/edit#pako:eNqNks9uhCAQxl_FzLFRg6sCcuilvfYFWhpDhLimIhsWt7s1vntdNtSuvZQLzPf9Zvg3EzRGKmBwdMKp5060VujktOOWD9Ey3h7eoyR5jHojZK2VNvYSrF-SRxrR97VeqvWBWJUVcMb0xzvAK_-roM7OisZtDnKvevAk-k4uF9qQG9mjn7b7w92i2lseWp4heGGGGFrbSWDOjioGrawW1xCmq8_B7ZVWHNiylMJ-cODDvOQcxPBqjA5p1oztPgTjQa6f8EOoQSr7ZMbBAaPEVwA2wRlYhklKy6LISpxRjEqcx3ABlmcpoQTnCCGCsxyTOYYvvydKaVXmNKsoQVVBSbWLQcnOGfty6wLfDPM3vDirhA)

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

    # 長期記憶から読み込んだ情報
    memories: list[str]

    # 今回の会話から抽出された記憶候補
    memory_candidates: list[str]

    # 保存してよいと判定された記憶
    approved_memories: list[str]

In [ ]:
from typing import Literal
from dataclasses import dataclass
from uuid import uuid4

from langchain_core.messages import AIMessage

from langgraph.runtime import Runtime


@dataclass
class Context:
    user_id: str

def load_memory(state: AgentState, runtime: Runtime[Context]) -> AgentState:
    """
    長期記憶をStoreから読む。
    現在のユーザー入力に意味的に関連する記憶だけをセマンティック検索で取得する。
    """
    user_id = runtime.context.user_id
    namespace = ("memories", user_id)

    # 直近のユーザー発話を検索クエリにする
    query = ""
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage) and msg.content:
            query = str(msg.content)
            break

    # プロンプトに載せる関連記憶の最大件数
    TOP_K = 5

    if query:
        try:
            # Storeに埋め込みindexがあれば関連度順に取得される
            items = runtime.store.search(namespace, query=query, limit=TOP_K)
        except Exception:
            # index未設定などで検索できない場合は全件取得にフォールバック
            items = runtime.store.search(namespace)
    else:
        # クエリが無いターンでは検索せず全件（もしくは無し）を返す
        items = runtime.store.search(namespace)

    return {
        "memories": [item.value["text"] for item in items],
    }


def call_model(state: AgentState) -> AgentState:
    """
    通常のReAct用LLM node。
    ここでは長期記憶をプロンプトに差し込む。
    """

    memory_text = "\n".join(f"- {m}" for m in state["memories"])

    system = SystemMessage(
        content=f"""
あなたはAIエージェントです。
必要ならツールを使ってください。

参考になる長期記憶:
{memory_text}
"""
    )

    response = model_with_tools.invoke(
        [system] + state["messages"]
    )

    return {
        "messages": [response],
    }


from pydantic import BaseModel, Field


class MemoryItem(BaseModel):
    """長期記憶として保存する価値のある、ユーザーに関する1件の事実。"""

    text: str = Field(
        description="三人称・簡潔・自己完結した日本語の事実文。例: 'ユーザーはPythonが好き'"
    )


class MemoryExtraction(BaseModel):
    """会話から抽出した長期記憶の候補一覧。"""

    candidates: list[MemoryItem] = Field(
        default_factory=list,
        description="保存候補のリスト。該当が無ければ空にする。",
    )


# 構造化出力でLLMから候補を取り出す抽出器
_memory_extractor = model.with_structured_output(MemoryExtraction)

_MEMORY_EXTRACT_SYSTEM = """あなたは会話から「長期記憶として保存する価値のある情報」だけを抽出する抽出器です。

抽出する: ユーザーの恒常的な好み・プロフィール・目標・制約・重要な決定など、将来の会話でも役立つ事実。
抽出しない: 一時的な依頼、計算やツールの実行結果、その場限りの話題、アシスタント自身の発言、単なる推測。

各候補は三人称・簡潔・自己完結した日本語の事実文にする（例:「ユーザーはPythonが好き」）。
該当が無ければ空のリストを返す。"""


def extract_memory(state: AgentState) -> AgentState:
    """
    今回の会話から、保存候補をLLMで抽出する。
    ここはLLMを使うが「ツール」ではなく、アプリ側が必ず呼ぶ後処理node。
    構造化出力を使い、保存に値する恒常的な事実だけを取り出す。
    """

    # 抽出対象は人間とアシスタントの発話テキストに限定する
    transcript_lines = []
    for msg in state["messages"]:
        if isinstance(msg, HumanMessage) and msg.content:
            transcript_lines.append(f"User: {msg.content}")
        elif isinstance(msg, AIMessage) and msg.content:
            transcript_lines.append(f"Assistant: {msg.content}")

    # 発話が無ければLLMを呼ばずに早期return
    if not transcript_lines:
        return {"memory_candidates": []}

    transcript = "\n".join(transcript_lines)

    try:
        result = _memory_extractor.invoke(
            [
                SystemMessage(content=_MEMORY_EXTRACT_SYSTEM),
                HumanMessage(
                    content=f"次の会話から長期記憶の保存候補を抽出してください。\n\n{transcript}"
                ),
            ]
        )
        candidates = [c.text.strip() for c in result.candidates if c.text.strip()]
    except Exception:
        # 抽出に失敗した場合は安全側に倒し、何も保存候補にしない
        candidates = []

    return {
        "memory_candidates": candidates,
    }


import re

# 長期保存すべきでないセンシティブ情報（PII等）を検出する正規表現
_SENSITIVE_PATTERNS = [
    re.compile(r"\b(?:\d[ -]?){13,16}\b"),                 # クレジットカード番号
    re.compile(r"\b\d{2,4}-\d{2,4}-\d{3,4}\b"),            # 電話番号など
    re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"),               # メールアドレス
    re.compile(r"パスワード|password|秘密の質問|暗証番号", re.I),  # 認証情報
]



def _normalize(text: str) -> str:
    """重複判定用に表記ゆれを吸収して正規化する。"""
    return re.sub(r"\s+", "", text).strip().lower()


def is_sensitive(text: str) -> bool:
    """保存すべきでないセンシティブ情報（PII・認証情報）を含むか判定する。"""
    return any(p.search(text) for p in _SENSITIVE_PATTERNS)



def validate_memory(state: AgentState) -> AgentState:
    """
    保存候補を検証する。
    ポリシー（センシティブ情報）・重複をチェックし、
    保存してよいものだけを approved_memories に残す。
    """

    approved = []

    # 既存記憶を正規化して重複排除のキー集合にする
    seen = {_normalize(m) for m in state["memories"]}

    for candidate in state["memory_candidates"]:
        key = _normalize(candidate)

        # 空・既存と重複・センシティブ・一時的なものは保存しない
        if not key or key in seen:
            continue
        if is_sensitive(candidate):
            continue

        approved.append(candidate)
        seen.add(key)  # 同一バッチ内での重複も防ぐ

    return {
        "approved_memories": approved,
    }


def write_memory(state: AgentState, runtime: Runtime[Context]) -> AgentState:
    """
    検証済みの記憶だけをStoreへ保存する。
    """

    user_id = runtime.context.user_id
    namespace = ("memories", user_id)

    for memory in state["approved_memories"]:
        runtime.store.put(
            namespace,
            str(uuid4()),
            {
                "text": memory,
                "source": "conversation",
            },
        )

    return {}


In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import OpenAIEmbeddings
from langgraph.store.memory import InMemoryStore
ltm_store = InMemoryStore(
    index={
        "embed": OpenAIEmbeddings(model="text-embedding-3-small"),
        "dims": 1536,
        "fields": ["text"],
    }
)

builder = StateGraph(AgentState)

builder.add_node(load_memory)
builder.add_node(call_model)
builder.add_node("tools", ToolNode(tools))
builder.add_node(extract_memory)
builder.add_node(validate_memory)
builder.add_node(write_memory)

builder.add_edge(START, "load_memory")
builder.add_edge("load_memory", "call_model")

builder.add_conditional_edges(
    "call_model",
    tools_condition,
    {
        "tools": "tools",
        END: "extract_memory",
    },
)

builder.add_edge("tools", "call_model")

builder.add_edge("extract_memory", "validate_memory")
builder.add_edge("validate_memory", "write_memory")
builder.add_edge("write_memory", END)

graph_with_long_term_memory = builder.compile(store=ltm_store)

In [ ]:
for chunk in graph_with_long_term_memory.stream(
    {
        "messages": [
            HumanMessage(
                content="覚えて: 私はPythonが好きです"
            )
        ]
    },
    context=Context(user_id="user-001"),
    #config = config
):
    print(chunk)
    print("-" * 80)

---
# 6. 記憶可能なAIエージェント: memonic-agent

- `graph_with_long_term_memory`をパッケージ化。
- ライブラリとして利用する他、LangGraphサーバー(APIサーバー)としても使用可能

In [ ]:
# === Google Colab で実行する場合のみ、次の2行の先頭の # を外して実行 
!git clone https://github.com/cosmac-dev/ai-agent-seminar.git
%cd ai-agent-seminar/session05
# セッションの再起動が要求されたら再起動後にOPENAI_API_KEYを再設定する

%pip install -e ".[server]"
!echo "OPENAI_API_KEY=$OPENAI_API_KEY" > .env

## 6.1 ライブラリとして使用

In [ ]:
from mnemonic_agent import Agent

agent = Agent()  # 既定で ChatOpenAI と埋め込み付き InMemoryStore を使用

agent.run("覚えて: 私はPythonが好きです", user_id="user-001", session_id="user-001:s1")
print(agent.run("私について覚えていることは？", user_id="user-001", session_id="user-001:s1"))
print(agent.recall("user-001"))  # 保存済みの長期記憶を一覧

## 6.2 LangGraphサーバーで動かす

In [ ]:
%cd /content/ai-agent-seminar/session05
!langgraph dev
# Colab環境では--tunnelオプションを付ける
# !langgraph dev --tunnel